In [8]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

gemini_model = init_chat_model(
    "gemini-3.1-flash-lite",
    model_provider="google_genai",
    temperature=1,         
    max_tokens=32000,      
    thinking_level="high", 
)
open_router_model = init_chat_model(
    "nvidia/nemotron-3-super-120b-a12b:free",
    model_provider="openrouter",
    temperature=0,
    max_tokens=5000,
)

gemini_model.invoke("hi")

AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'Et0ECtoEARFNMg9zVN/FHmsEZwFKkyqTgXe5bPsTIwC39Vi4GJr4BB5t/39JCgRkIQdsQapZLg7GYj2h+rCllCNfQoB/ISpMu07+R2YCQ8hX6m1Us4fJ3blwi+ZVQBM+koWDCoCP8/yV4TJxz8AF2iFaHu+5Y/TdvUIqzPJ0EhLYFQ4NOefhRgCCWkS3XtA8JJR0DWwIfSKdB44eOEBVZ9/R4t96itoEB2mNjbDtMTSRanbl58vS7S7NWkhXw+mkCT2CRIcAUreML2t0YtLskXujgCSs12RWCfYkA19eBkrnFT45QBJALvSjvfFzRWLLLF248Y/6cdPvyuwbBELdEzvPXc96plr1hQ83LIZvhyiYddl3qOoOjlRv8BBdKYm0c7t1NGrEWtMnBwRSgoVPm3ps0HHy30GxnmNgG+aMzZEQ035++5z+MDLo6s/MOvOKleHFsrIDVoQWRy6onTKNxv8UZyJaYsFyULChXyoGFgQVmdFwcdaavK2KX6OJjZocUfbW/XiixH7vf+Qy1IN8vn4i6zQkt5h+1weNq7v3eBJr720xxY3aTPLeAKFNMiWLkHyje5L4tBVUoGXfuUriBBQV+oFR45vLysS4w/+chR1inGBlpm+eLxYIchAzFNRRzUFXTX9JDEPyirEeA4ND64V6mJIkahqB9InVagBudk11+iePsQvdpH0GS4cyQbjFK05JFkwsReCBFM+47AwgaPWz5LBIpncQlpVxNX5IorVo44b2I4sv5xGuLHbJ+0TvfuFTGPrkwLFnihGZ4cgCbPfEqpMz3pS6rS/m0UV7g8M='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name

In [9]:
#initialising states
from typing import TypedDict, Annotated, List
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage


class AgentState(TypedDict):    
    messages: Annotated[List[BaseMessage], add_messages]

In [10]:
#load all csvs into duckdb and pass in the table names
import os
import duckdb

_connection = None


def get_connection() -> duckdb.DuckDBPyConnection:
    global _connection
    if _connection is None:
        _connection = duckdb.connect(database=":memory:")
    return _connection


def load_tables(data_dir) -> list[str]:
    """Register every CSV file in data_dir as a DuckDB table, named after
    the file (without extension). Returns the table names in the order
    they were loaded."""
    con = get_connection()
    table_names = []
    for filename in sorted(os.listdir(data_dir)):
        if not filename.endswith(".csv"):
            continue
        table_name = os.path.splitext(filename)[0]
        path = os.path.join(data_dir, filename)
        con.execute(
            f'CREATE OR REPLACE TABLE "{table_name}" AS '
            f"SELECT * FROM read_csv_auto(?)",
            [path],
        )
        table_names.append(table_name)
    return table_names

In [14]:
from langchain.tools import tool
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent
DATA_DIR = BASE_DIR / "data" / "synthea_dataset"

@tool
def extract_all_metadata():
    """Extracts metadata from the given table"""
    return None

@tool
def infer_composite_pk():
    """Infers composite PK from extracted metadata"""
    return None

@tool
def infer_simple_pk():
    """Infers simple PK from extracted metadata"""
    return None

@tool
def create_rule_plan():
    """Creates Data Quality Rules list that must be checked"""
    return None

@tool
def generate_sql():
    """Generate SQL queries"""
    return None

@tool
def validate_sql():
    """Validates SQL queries"""
    return None

@tool
def execute_sql():
    """Executes SQL queries"""
    return None

@tool
def write_report():
    """Writes Data Quality Report (rule, sql query, output, inference)"""
    return None

NameError: name '__file__' is not defined

In [15]:
from langchain.agents import create_agent
from deepagents.middleware.filesystem import FilesystemMiddleware
from langchain.agents.middleware import TodoListMiddleware

tools=[extract_all_metadata, infer_composite_pk, infer_simple_pk, create_rule_plan, generate_sql, validate_sql, execute_sql, write_report]
SYSTEM_PROMPT="""<role>
You are the orchestrator of a data quality (DQ) pipeline. Each time you run, you are working on exactly ONE table. You operate using a single message timeline, relying entirely on your Todo List to track your progress and the Filesystem to manage data. 
</role>

<context>
Other tables in this database may already have been checked in earlier, separate runs—their results are already stored on the filesystem. When you generate this table's final results, use your filesystem tools to append them to the shared report on disk. Do not start a new report, and you do not need to know about any other table.
</context>

<workflow>
Leverage your Todo List to manage your execution. Follow this strict lifecycle:

1. Metadata Extraction: Call `extract_all_metadata` to profile the columns. This deterministic tool will return the schema, row count, sample rows, column profile stats (e.g., null counts, distinct ratios), and identify a simple Candidate Key (CK) if one is present.
2. Primary Key (PK) Inference: Use the extracted metadata (schema, row_count, sample_rows, CKs, and profile_columns_stats) to deduce the PK:
    - If a simple CK is found: Call `infer_simple_pk`.
    - If no simple CK is found: Call `infer_composite_pk`.
    - Note: If you are evaluating a Department table, always enforce that 'DeptName' is the primary key rather than just a candidate key.
3. Plan: Once the final PK is returned, use this context to populate your Todo List with the required data quality checks for this table using `create_rule_plan`.
4. Generate & Validate: For each pending check on your Todo List, call `generate_sql`. Immediately call `validate_sql` afterward. 
5. Self-Correct: If `validate_sql` reports failures, retry `generate_sql` for that specific check until valid (or until you are told retries are exhausted).
6. Execute & Track: Call `execute_sql` for the valid checks. As each check finishes executing, mark it as complete on your Todo List.
7. Report: Once your Todo List is completely clear of pending rule checks, write the final results to the filesystem using `write_report`. This is always the last step, called exactly once for this table.
</workflow>

<stop_condition>
Once the report is successfully appended to the filesystem and your Todo List is empty, respond with a short plain-text summary to the user. Do not call any further tools; this ends the run.
</stop_condition>
"""

model=create_agent(
    model=gemini_model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    middleware=[FilesystemMiddleware(), TodoListMiddleware()]
)

In [16]:
if __name__=="__main__":
    table_names=load_tables(DATA_DIR)
    for table in table_names:
        model.invoke("Run the Data Quality check.")

NameError: name 'DATA_DIR' is not defined